# v8 Lecture: Perelman–Ricci Theory to Finance

**Graduate lecture notebook.**

This v8 version extends the v7 rolling Ricci/HMM market-network model by adding **Ricci flow**. Curvature measures local network geometry; Ricci flow then deforms financial distances so coherent clusters contract and fragile bridge-like links stretch.

> Mathematical analogy: Perelman used Ricci flow to reveal and simplify geometric structure. In finance, we use a graph version of Ricci flow as a diagnostic lens for market topology, not as a direct price predictor.

## Learning goals

After this lecture, students should be able to:

1. Build a correlation-distance graph from asset returns.
2. Interpret Ollivier-Ricci curvature on financial networks.
3. Use rolling curvature features to describe market regimes.
4. Explain why Ricci flow is useful: it amplifies hidden structure by shrinking robust/coherent edges and stretching fragile/stress edges.
5. Run and critique a small empirical or synthetic experiment.

In [ ]:
# Core imports
import sys
sys.path.append('.')

import pandas as pd
import numpy as np

from helper import (
    DEFAULT_TICKERS, make_demo_prices, download_prices, prices_to_returns,
    build_rolling_frames, build_base_graph_for_layout, compute_stable_layout,
    visualize_network, rolling_feature_table, plot_rolling_features,
    compute_hmm_regimes, summarize_edges, run_ricci_flow,
    plot_ricci_flow_history, compare_before_after_flow
)

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 20)

## 1. Data: synthetic first, live market data optional

For class reliability, start with synthetic prices. Replace this block with `download_prices(...)` when internet/yfinance is available.

In [ ]:
tickers = ["NVDA", "AMD", "AVGO", "TSM", "MU", "LRCX", "KLAC", "ANET", "AAOI", "IONQ"]

# Offline lecture data
prices = make_demo_prices(tickers=tickers, n_days=300, seed=8)

# Optional live data:
# prices = download_prices(tickers, period="1y", interval="1d")

returns = prices_to_returns(prices)
prices.tail()

## 2. Financial distance graph

Given correlation $\rho_{ij}$, define a financial distance

$$d_{ij}=\sqrt{2(1-\rho_{ij})}.$$

Small distance means two assets move together. We then threshold this graph so only economically meaningful links remain.

In [ ]:
frames, starts = build_rolling_frames(
    returns,
    window_size=60,
    step=10,
    max_frames=30,
    max_distance=1.10,
    min_abs_corr=0.25,
    keep_top_edges=35,
    alpha=0.5,
    method="OTD",
    proc=1,
)

features = rolling_feature_table(frames)
features.tail()

In [ ]:
base = build_base_graph_for_layout(frames, all_nodes=returns.columns)
pos = compute_stable_layout(base, seed=42)

last = frames[-1]
fig = visualize_network(last.G, positions=pos, node_cluster=last.node_cluster, title="Latest rolling Ricci financial network")
fig.show()

## 3. Curvature interpretation

For an edge $i-j$:

- **Positive Ricci curvature**: the two neighborhoods overlap or move coherently. In finance, this often indicates stable cluster structure.
- **Negative Ricci curvature**: the edge is bridge-like; neighborhoods disagree. In finance, this can indicate fragility, rotation, contagion path, or regime boundary.
- **Average curvature** over time is a market-topology observable, not a trading signal by itself.

In [ ]:
summarize_edges(last.G).head(12)

In [ ]:
plot_rolling_features(features).show()

## 4. Hidden Markov model regimes from Ricci features

The HMM is used only as an unsupervised regime labeler. It sees features such as average Ricci curvature, edge density, cluster count, and largest-component ratio.

Important implementation note: `hmmlearn.GaussianHMM` does **not** provide `fit_predict` in some versions, so v8 uses:

```python
model.fit(X)
states = model.predict(X)
```

In [ ]:
hmm_df, regime_names = compute_hmm_regimes(
    frames, returns, starts, window_size=60, n_components=3, forward_days=5, random_state=42
)
regime_names, hmm_df.tail()

## 5. v8 upgrade: Ricci flow

Ricci curvature tells us what the local network geometry looks like. Ricci flow asks: **what happens if the network is allowed to deform according to its own curvature?**

For each edge distance $w_{ij}$, v8 uses the discrete graph-flow update

$$w_{ij}^{(t+1)} = w_{ij}^{(t)}\left(1 - \eta \kappa_{ij}^{(t)}\right),$$

where $\eta$ is the step size and $\kappa_{ij}$ is Ricci curvature.

### Why this matters in finance

- Positive-curvature edges shrink: coherent sector/theme links become tighter.
- Negative-curvature edges stretch: fragile bridges and possible stress-transmission channels become more visible.
- Flow can reveal whether a market network has stable clusters, unstable bridges, or a single crowding/risk-on component.
- It is a geometry stress test: useful for portfolio diversification, regime diagnostics, and contagion analysis.

In [ ]:
G0 = last.G.copy()
G_flow, flow_history = run_ricci_flow(
    G0,
    iterations=10,
    step_size=0.20,
    alpha=0.5,
    method="OTD",
    proc=1,
    normalize_mean_weight=True,
)
flow_history

In [ ]:
plot_ricci_flow_history(flow_history).show()

In [ ]:
fig_before = visualize_network(G0, positions=pos, title="Before Ricci flow")
fig_before.show()

fig_after = visualize_network(G_flow, positions=pos, title="After Ricci flow: contracted coherent links, stretched fragile links")
fig_after.show()

## 6. Edge-level flow diagnosis

Read `distance_change` as a deformation score:

- Negative `distance_change`: the edge contracted; the relationship is geometrically coherent.
- Positive `distance_change`: the edge stretched; the relationship behaves more like a bridge or stress link.

In [ ]:
flow_compare = compare_before_after_flow(G0, G_flow)
flow_compare.head(15)

In [ ]:
flow_compare.tail(15)

## 7. Graduate discussion prompts

1. How does Ricci curvature differ from correlation?
2. Why might a highly correlated pair still be fragile in a network sense?
3. What is the risk of using Ricci flow as a trading signal instead of a structural diagnostic?
4. How could you validate whether negative-curvature bridges predict future volatility or drawdown?
5. What changes if the nodes are sectors, ETFs, supply-chain firms, or options-implied volatility surfaces?

## 8. Suggested assignment

Choose 15–30 assets from one theme, such as AI semiconductors, memory, quantum, optical networking, or ETFs.

1. Build rolling Ricci networks.
2. Plot average curvature, density, and cluster count.
3. Run HMM regimes.
4. Run Ricci flow on one calm window and one stress window.
5. Explain which links contract, which links stretch, and what that implies for diversification or contagion.